In [ ]:
#| default_exp fid

In [ ]:
# this notebook is origianlly called 18_fid.ipynb in the fast_ai course repo
# i changed the name because it's not only about FID.
# however, for future compatibility of modules i keep the module name as fid.

# Load deps

In [ ]:
# ! pip install -q pytorch_fid torcheval

In [ ]:
# # if src modules imported
# from google.colab import drive
# drive.mount('/content/drive')
import sys
app_path = '/content/drive/MyDrive/Projects/miniSD'
sys.path.append(app_path)

# and upload the local artifacts/ dir to colab
# it includes required pretrained checkpoints

In [ ]:
#|export
import math,torch, os
from tqdm.auto import tqdm
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy import linalg
import torch.nn.functional as F
import torchvision.transforms.functional as TF
from torch import tensor,nn, distributions
from datasets import load_dataset
from pytorch_fid.inception import InceptionV3
from functools import partial
from diffusers import UNet2DModel

from src.utils import set_seed, noop
from src.datasets import inplace, DataLoaders, show_images
from src.learner import DeviceCB, to_cpu, TrainLearner
from src.activations import HooksCallback
from src.conv import def_device
from src.init import GeneralRelu, init_weights
from src.resnet import ResBlock
from src.augment import capture_preds
# TODO: we use this import to patch the capture_preds to the `Learner` class
# this is not a good practice at all. change it ASAP

# Config

In [ ]:
torch.set_printoptions(precision=2, linewidth=140, sci_mode=False)
torch.manual_seed(1)
mpl.rcParams['image.cmap'] = 'gray_r'
set_seed(42)
num_dl_workers = os.cpu_count()
dataset_xl,dataset_yl = 'image','label'
dataset_name = "fashion_mnist"
bs = 128

# Load dataset

In [ ]:
@inplace
def transformi(b):
    b[dataset_xl] = [
        F.pad(TF.to_tensor(o), (2,2,2,2))*2-1
        for o in b[dataset_xl]
    ]

dsd = load_dataset(dataset_name)
tds = dsd.with_transform(transformi)
dls = DataLoaders.from_dd(tds, bs, num_workers=num_dl_workers)
b = xb,yb = next(iter(dls.train))

# Load the pretrained model (the hard way!!!)

In [ ]:
# TODO: change the saving method so we don't have to define all this to load the model!!!
class Dropout(nn.Module):
    def __init__(self, p=0.1):
        super().__init__()
        self.p = p

    def forward(self, x):
        if not self.training: return x
        dist = distributions.binomial.Binomial(tensor(1.0).to(x.device), probs=1-self.p)
        return x * dist.sample(x.size()) * 1/(1-self.p)

def get_dropmodel(
    act=nn.ReLU,
    nfs=(16,32,64,128,256,512),
    norm=nn.BatchNorm2d,
    drop=0.0
):
    layers = [
        ResBlock(1, 16, ks=5, stride=1, act=act, norm=norm),
        nn.Dropout2d(drop)
    ]
    layers += [
        ResBlock(nfs[i], nfs[i+1], act=act, norm=norm, stride=2)
        for i in range(len(nfs)-1)
    ]
    layers += [
        nn.Flatten(),
        Dropout(drop),
        nn.Linear(nfs[-1], 10, bias=False),
        nn.BatchNorm1d(10)
    ]
    return nn.Sequential(*layers)

In [ ]:
act_gr = partial(GeneralRelu, leak=0.1, sub=0.4)
iw = partial(init_weights, leaky=0.1)
model = get_dropmodel(act_gr, norm=nn.BatchNorm2d, drop=0.1).apply(iw)
loaded_art = torch.load('artifacts/models/data_aug2.pkl', weights_only=False)
model.load_state_dict(loaded_art.state_dict())

# How to extract the features?
we need to extract feature based on which we want to run statistical tests. we have two options:
- using a Hook to record a module's activations
- delete all modules after the desired one and capture the output of the model

In [ ]:
learn = TrainLearner(model, dls, cbs=[DeviceCB()], loss_func=noop)

## Using Hook CB

In [ ]:
def append_outp(hook, mod, inp, outp):
    if not hasattr(hook,'outp'): hook.outp = []
    hook.outp.append(to_cpu(outp))
hcb = HooksCallback(append_outp, mods=[learn.model[6]], on_valid=True)
learn.fit(1, train=False, cbs=[hcb])
feats = torch.cat(hcb.hooks[0].outp).float().squeeze()
print(feats.shape)

## Using capture predictions

In [ ]:
learn.summary()

In [ ]:
# we want to skip last three layers.
# keeping flatten is optional since we squeeze the tensor below.
model = model[:-3]
learn.model = model
feats,y = learn.capture_preds()
# keep_loss=True works too because of loss_func=noop
# two ways to bypass loss calulation:
# 1-capture_preds set get_loss by default keep_loss=False
# 2-use loss_func=noop when instantiating the learner
feats = feats.float().squeeze()
print(feats.shape,y)

- So far, we calculated the features representation for **real images**.
- Next, we do the same for **sampled images** from our `smodel`

# Extract features of the sampled images

In [ ]:
class UNet(UNet2DModel):
    def forward(self, x): return super().forward(*x).sample

betamin,betamax,n_steps = 0.0001,0.02,400
beta = torch.linspace(betamin, betamax, n_steps)
alpha = 1.-beta
alphabar = alpha.cumprod(dim=0)
sigma = beta.sqrt()
smodel = UNet(
    in_channels=1,
    out_channels=1,
    block_out_channels=(16, 32, 64, 128),
    norm_num_groups=8
)
loaded_art = torch.load('artifacts/models/fashion_ddpm_mp.pkl', weights_only=False, map_location=def_device)
smodel.load_state_dict(loaded_art.state_dict())

In [ ]:
@torch.no_grad()
def sample(model, sz, alpha, alphabar, sigma, n_steps):
    device = next(model.parameters()).device
    x_t = torch.randn(sz, device=device)
    preds = []
    for t in tqdm(reversed(range(n_steps)), total=n_steps):
        t_batch = torch.full((x_t.shape[0],), t, device=device, dtype=torch.long)
        z = (torch.randn(x_t.shape) if t > 0 else torch.zeros(x_t.shape)).to(device)
        ᾱ_t1 = alphabar[t-1]  if t > 0 else torch.tensor(1)
        b̄_t = 1 - alphabar[t]
        b̄_t1 = 1 - ᾱ_t1
        x_0_hat = ((x_t - b̄_t.sqrt() * model((x_t, t_batch)))/alphabar[t].sqrt())
        x_t = x_0_hat * ᾱ_t1.sqrt()*(1-alpha[t])/b̄_t + x_t * alpha[t].sqrt()*b̄_t1/b̄_t + sigma[t]*z
        preds.append(x_0_hat.cpu())
    return preds

- smodel, which is based on fashion_ddpm_mp.pkl was trained with images whose values were in [0,1]
- to see this look at the Load data section for that model where we only used `TF.to_tensor` that maps values to [0,1]
- the model, which is based on data_aug2.pkl was trained with images whose values were in [-1,1]
- Hence, in order to match value ranges we map our samples from [0,1] to [-1,1]
- TODO: this is just a temporary and partial fix. the correct fix is to train both models with images of the same input range

In [ ]:
samples = sample(smodel, (128, 1, 32, 32), alpha, alphabar, sigma, n_steps)
stacked_samples = torch.stack(samples).contiguous().clone().detach()
from pathlib import Path
data_path = Path("artifacts/data")
data_path.mkdir(exist_ok=True)

# torch.save(stacked_samples, data_path / "fashion_ddpm_mp_samples.pt")
# torch.save(samples[-1].clone().detach(), data_path / "fashion_ddpm_mp_samples_last.pt")

In [ ]:
# samples_last = torch.load(data_path / "fashion_ddpm_mp_samples_last.pt")
# s = samples_last*2-1
s = samples[-1]*2-1
show_images(s[:16], imsize=1.5);

In [ ]:
clearn = TrainLearner(
    model,
    DataLoaders([],[(s,yb)]), # it doesn't matter what the dependent variable/`yb` is.
    cbs=[DeviceCB()],
    loss_func=noop
)

feats2,y2 = clearn.capture_preds()
feats2 = feats2.float().squeeze()
print(feats2.shape)

# Calc Fréchet Inception Distance (FID) metric

- The model we're evaluating is `smodel`. it's based on `"models/fashion_ddpm_mp.pkl"` checkpoint
- The other `model` which is based on `"models/data_aug2.pkl"` is assumed to be the reference model
- The **inception** in the name comes from the fact that this metric was initially used with the **google/inception** model as the reference model
- We are using our own model as the reference model here, which we believe is good at classifying the fashion-mnist dataset (this is the important asuumtpion)
- FID Caveats:
    - This metric is biased against the number of samples, meaning that if we use less samples the FID would be higher on average. WHY??
        - papers report FID alongside the # samples to be comparable to other models
    - In order to use this metric you have to resize the input images to the required input size of the reference model and this might cause the image lose details or quality
        - google/inceptionV3 expects 299x299 input images for example.

In [ ]:
means = feats.mean(0)
print(means.shape)
covs = feats.T.cov()
print(covs.shape)

In [ ]:
#|export
def _calc_stats(feats):
    feats = feats.squeeze()
    return feats.mean(0),feats.T.cov()

def _sqrtm_newton_schulz(mat, num_iters=100):
    mat_nrm = mat.norm()
    mat = mat.double()
    Y = mat/mat_nrm
    n = len(mat)
    I = torch.eye(n, n).to(mat)
    Z = torch.eye(n, n).to(mat)

    for i in range(num_iters):
        T = (3*I - Z@Y)/2
        Y,Z = Y@T,T@Z
        res = Y*mat_nrm.sqrt()
        if ((mat-(res@res)).norm()/mat_nrm).abs()<=1e-6: break
    return res

def _calc_fid(m1,c1,m2,c2):
    # csr = _sqrtm_newton_schulz(c1@c2)
    csr = tensor(linalg.sqrtm(c1@c2).real)
    return (((m1-m2)**2).sum() + c1.trace() + c2.trace() - 2*csr.trace()).item()

In [ ]:
s1,s2 = _calc_stats(feats),_calc_stats(feats2)
fid_score = _calc_fid(*s1, *s2)
print(f"fid_score: {fid_score}")

# Calc Kernel Inception Distance (KID) metric
- this metric is not biased by the number of samples
- it's easier to calculate than the FID metric

In [ ]:
#|export
def _squared_mmd(x, y):
    def k(a,b): return (a@b.transpose(-2,-1)/a.shape[-1]+1)**3
    m,n = x.shape[-2],y.shape[-2]
    kxx,kyy,kxy = k(x,x), k(y,y), k(x,y)
    # it's function of the kernel calculation
        # between each sampela and itself
        # and across samples
    kxx_sum = kxx.sum([-1,-2])-kxx.diagonal(0,-1,-2).sum(-1)
    kyy_sum = kyy.sum([-1,-2])-kyy.diagonal(0,-1,-2).sum(-1)
    kxy_sum = kxy.sum([-1,-2])
    return kxx_sum/m/(m-1) + kyy_sum/n/(n-1) - kxy_sum*2/m/n

def _calc_kid(x, y, maxs=50):
    xs,ys = x.shape[0],y.shape[0]
    n = max(math.ceil(min(xs/maxs, ys/maxs)), 4)
    # partitions data
    mmd = 0.
    for i in range(n):
        cur_x = x[round(i*xs/n) : round((i+1)*xs/n)]
        cur_y = y[round(i*ys/n) : round((i+1)*ys/n)]
        mmd += _squared_mmd(cur_x, cur_y)
        # calculates and aggregates the kernel metric across partitions
    return (mmd/n).item()

In [ ]:
print(feats.shape, feats2.shape)
_calc_kid(feats, feats2)

# TODO: what's the scale of this metric? what are high and low values?

- KID caveat: it has a **high variance** across different samples and seeds
- Therefore, the current situation is that currently we don't have an unbiased, consistent and efficient metric to use.
    - What is the solution???

# Image generation evaluation class

In [ ]:
#|export
class ImageEval:
    def __init__(self, model, dls, cbs=None):
        # this class assumes that the desired feature space is generated by the model
        # so, handling model layers and heads should be done by the user.
        self.learn = TrainLearner(model, dls, loss_func=noop, cbs=cbs)
        self.feats = self.learn.capture_preds()[0].float().cpu().squeeze()
        self.stats = _calc_stats(self.feats)

    def get_feats(self, samp):
        self.learn.dls = DataLoaders([],[(samp, tensor([0]))]) # any y value is fine here
        return self.learn.capture_preds()[0].float().cpu().squeeze()

    def fid(self, samp): return _calc_fid(*self.stats, *_calc_stats(self.get_feats(samp)))
    def kid(self, samp): return _calc_kid(self.feats, self.get_feats(samp))

In [ ]:
ie = ImageEval(model, learn.dls, cbs=[DeviceCB()])

In [ ]:
print(ie.fid(s))
print(ie.kid(s))

## FID and KID over time (sampling timestep)
- the images are the predicted x_0 at each time step (all the noise is subtracted)

In [ ]:
xs = list(range(0,n_steps,20))+[n_steps - 11, n_steps - 2]
ys_fid = [ie.fid(samples[i].clamp(-0.5,0.5)*2) for i in xs]
ys_kid = [ie.kid(samples[i].clamp(-0.5,0.5)*2) for i in xs]

In [ ]:
plt.plot(xs, ys_fid, label="FID");
plt.plot(xs, ys_kid, label="KID");
plt.yscale("log")
plt.legend()

## Calculation for a batch of real data
- Somehow, measuring how good we could get
    - make sure to take the sample size into account when using FID

In [ ]:
print(ie.fid(xb))
print(ie.kid(xb))
# TODO: what does negative KID mean?

# Calculting the standard FID
- using the google/inception model
- it expects `(C, H, W) = 3x299x299` input images

In [ ]:
# a = tensor([1,2,3])
# print(a)
# print(a.repeat((3,1)))

In [ ]:
class IncepWrap(nn.Module):
    def __init__(self):
        super().__init__()
        self.m = InceptionV3(resize_input=True)
        # TODO: instead of importing this model, implement it from scratch
        # basically, replace its checkpoint with the data_aug2 checkpoint.
    def forward(self, x):
        return self.m(x.repeat(1,3,1,1))[0]
    # since we have only 1 channel, we replicate it 3 times

# tds = dsd.with_transform(transformi)
# dls = DataLoaders.from_dd(tds, bs, num_workers=num_dl_workers)
ie = ImageEval(IncepWrap(), dls, cbs=[DeviceCB()])

- One way to compare the inception model with our own model (data_aug2 checkpoint) is
    - comparing the ratio between sample metrics and real image bacth metrics (shown below)
    - the metrics value on real data
        - lower FID and KID shows that our model is more competent at detecting real data

In [ ]:
print(ie.fid(s))
print(ie.kid(s))

In [ ]:
print(ie.fid(xb))
print(ie.kid(xb))